# Pretraining a 27M-parameter language model, step by step

This notebook walks through the whole TinyLM pipeline, from raw TinyStories text to a trained,
evaluated and serialized model. It is the **narrative** companion to the repository: every cell
calls the `tinylm` package, the same code the CLI scripts run, so nothing here is notebook-only
logic.

**What is trained from scratch, precisely:** the tokenizer and all model weights start from
random initialization. The transformer *architecture* is Hugging Face's `LlamaForCausalLM`; the
training loop, data pipeline and evaluation are in `src/tinylm/`. A separate from-scratch
PyTorch implementation of every component is in `src/tinylm/educational/` (Part 3 shows that it
reproduces the Hugging Face model's logits exactly).

| Part | Topic |
|---|---|
| 0 | Environment, configuration, run mode |
| 1 | Data: loading, normalization, the train / validation / test protocol |
| 2 | Tokenizer training and sequence packing with exact token accounting |
| 3 | The model: architecture, parameter budget, initialization check, educational re-implementation |
| 4 | Pretraining: the loop, checkpoints, and resuming correctly |
| 5 | Memory and throughput |
| 6 | Evaluation: held-out test loss, generation, and what perplexity does *not* tell you |
| 7 | Saving, reloading and serving the model |

## Part 0: Environment

On Colab, the first cell clones the repository and installs it. Locally, run the notebook from a
`uv sync` environment instead (see the README).

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists("tiny-language-models"):
    subprocess.run(
        ["git", "clone", "https://github.com/sudhanshumukherjeexx/tiny-language-models"], check=True
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", "tiny-language-models"], check=True
    )
if IN_COLAB:
    os.chdir("tiny-language-models")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # run from the repository root so config paths resolve
print("working directory:", os.getcwd())

In [ ]:
import json

import numpy as np
import torch

from tinylm.utils import environment_info, resolve_device, resolve_precision, setup_logging

setup_logging()
device = resolve_device("auto")
env = environment_info(device)
print(
    json.dumps(
        {k: env[k] for k in ("python", "torch", "transformers", "hardware", "git_commit")}, indent=2
    )
)
print("precision used for training:", resolve_precision("auto", device))

**Precision.** `auto` picks bf16 on Ampere or newer GPUs (A100, L4, RTX 30xx/40xx), fp16 with loss
scaling on GPUs without bf16 support (the free-tier T4 is Turing), and fp32 on CPU. bf16 has
the same exponent range as fp32, so it needs no loss scaling. fp16 does.

### Choose a run mode

| `MODE` | Data | Steps | What it is for |
|---|---|---|---|
| `"smoke"` | 2,000 stories, tiny model | 60 | Checks the pipeline end to end in minutes, even on CPU. **Not a result.** |
| `"quick"` | 300,000 stories, full model | 4,000 | A first real model on a single GPU session |
| `"full"`  | all ~2.1M stories, full model | 12,000 | The main experiment (`configs/tiny-27m.yaml`) |

Each mode writes to its **own** data and run directories. The original version of this notebook
resumed a 4,000-step subset run as the 12,000-step full run in the same directory, which produced
a two-phase training curve (see `results/runs/legacy-colab-a100/README.md`). The trainer now
refuses to resume when the configuration has changed.

In [ ]:
from tinylm.config import load_config

MODE = "smoke"  # "smoke" | "quick" | "full"
PERSIST_DIR = None  # e.g. "/content/drive/MyDrive/tinylm" to keep data and checkpoints on Drive

overrides = []
if MODE == "smoke":
    cfg_path = "configs/smoke.yaml"
elif MODE == "quick":
    cfg_path = "configs/tiny-27m.yaml"
    overrides = [
        "name=tiny-27m-quick",
        "data.max_train_examples=300000",
        "data.data_dir=data/quick",
        "tokenizer.path=data/tokenizer-quick",
        "training.max_steps=4000",
        "training.output_dir=runs/tiny-27m-quick",
    ]
else:
    cfg_path = "configs/tiny-27m.yaml"

if PERSIST_DIR:
    from google.colab import drive  # only needed when persisting to Drive

    drive.mount("/content/drive")
    base = load_config(cfg_path, overrides)
    overrides += [
        f"data.data_dir={PERSIST_DIR}/{base.data.data_dir}",
        f"tokenizer.path={PERSIST_DIR}/{base.tokenizer.path}",
        f"training.output_dir={PERSIST_DIR}/{base.training.output_dir}",
    ]

cfg = load_config(cfg_path, overrides)
print(f"config: {cfg_path}  overrides: {overrides}")
print(
    f"model: {cfg.model.num_hidden_layers} layers, d={cfg.model.hidden_size}, vocab={cfg.model.vocab_size}"
)
print(f"run dir: {cfg.training.output_dir}")

---
## Part 1: Data

### Why TinyStories
A ~27M-parameter model cannot learn the distribution of web text. TinyStories (Eldan & Li, 2023)
is ~2.1M synthetic short stories written with a small vocabulary. On that narrow domain, small
models produce fluent text, so we can study coherence, memorization and failure modes instead of
stopping at "not grammatical yet". The price: nothing here measures world knowledge or general
language ability.

### Three splits, three roles
TinyStories ships only `train` and `validation`. We need three roles:

| Split | Source | Used for |
|---|---|---|
| **train** | official `train` minus the validation slice | gradient updates |
| **validation** | 0.5 % of official `train`, chosen by hashing each normalized story | checkpoint selection, ablations |
| **test** | official `validation` minus stories with an exact copy in train | **one** final evaluation of the selected model |

Hash-based assignment is deterministic and seedless, and exact duplicates always land in the same
split. Leakage is **measured**, not assumed away (`scripts/check_data_integrity.py`). It matters
here: 29.9 % of the official validation stories have an exact copy in the official train split.
The original notebook reported its validation loss on that split. Those stories are removed from
test (`data.decontaminate_test`), and the cell below prints how many.

In [ ]:
from pathlib import Path

from tinylm.data import build_text_splits

splits = build_text_splits(cfg.data)
print(splits)
text_meta = json.loads((Path(cfg.data.data_dir) / "text" / "text_meta.json").read_text())
print(
    "test stories removed as exact copies of training stories:",
    text_meta["test_removed_exact_copy_of_train"],
)
for i in range(2):
    print(f"\n--- train story {i} ---\n{splits['train'][i]['text'][:400]}")

In [ ]:
import matplotlib.pyplot as plt

sample = splits["train"].select(range(min(20_000, len(splits["train"]))))["text"]
lengths = np.array([len(t) for t in sample])
print(
    f"characters per story: mean {lengths.mean():.0f} | median {np.median(lengths):.0f} | p95 {np.percentile(lengths, 95):.0f}"
)
plt.figure(figsize=(8, 3))
plt.hist(lengths, bins=60, color="#3b6ea8")
plt.title("Characters per story (sample of training split)")
plt.xlabel("characters")
plt.tight_layout()
plt.show()

### Normalization: keep Unicode, normalize typography
The original notebook deleted every non-ASCII character. A byte-level BPE tokenizer can encode
any byte, so that step is not needed. It also silently corrupts words. We keep NFKC
normalization and map curly quotes, dashes and ellipses to ASCII (they are typographic variants
of the same symbol), and leave everything else intact. The legacy behaviour is still available as
`strip_non_ascii: true`.

In [ ]:
from dataclasses import replace

from tinylm.data import normalize_text

example = "Café “quote” — and…   spaces"
print("current :", normalize_text(example, cfg.data.normalization))
print("legacy  :", normalize_text(example, replace(cfg.data.normalization, strip_non_ascii=True)))

---
## Part 2: Tokenizer and packing

### Why train our own 8,192-token tokenizer
GPT-2's 50,257-token vocabulary would need a 50,257 × 512 ≈ 25.7M-parameter embedding table, as
large as the rest of this model, and most rows would never see a gradient on children's stories.
At 8,192 tokens, the (tied) embedding is 4.2M parameters. The cost is slightly more tokens per
story.

The tokenizer is trained on the **train split only**, then wrapped in `PreTrainedTokenizerFast`
so it works with `generate()`, `pipeline()` and `save_pretrained()`.

In [ ]:
from pathlib import Path

from tinylm.tokenizer import (
    iter_batches,
    load_tokenizer,
    tokenizer_diagnostics,
    tokenizer_fingerprint,
    train_tokenizer,
)

tok_dir = Path(cfg.tokenizer.path)
if (tok_dir / "tokenizer.json").exists():
    tokenizer = load_tokenizer(tok_dir)
    print("loaded existing tokenizer from", tok_dir)
else:
    tokenizer = train_tokenizer(
        iter_batches(splits["train"]["text"]),
        cfg.tokenizer,
        cfg.data.block_size,
        length=len(splits["train"]),
    )
    tokenizer.save_pretrained(str(tok_dir))
    tokenizer = load_tokenizer(tok_dir)  # always use the reloaded copy
print(
    f"vocab {len(tokenizer):,} | eos/bos id {tokenizer.eos_token_id} | pad id {tokenizer.pad_token_id} | unk {tokenizer.unk_token}"
)

diag = tokenizer_diagnostics(tokenizer, list(splits["validation"].select(range(1000))["text"]))
print(
    f"characters/token {diag['characters_per_token']:.2f} | tokens/story mean {diag['tokens_per_story_mean']:.0f}, "
    f"p95 {diag['tokens_per_story_p95']} | exact round-trip {diag['roundtrip_exact_fraction']:.0%}"
)
for p in diag["probes"]:
    print(f"{p['tokens']:>3} tokens  roundtrip={p['roundtrip_exact']}  {p['text']!r}")

### Packing, with every token accounted for
Each story is tokenized and followed by one EOS token, the stream is concatenated, and it is cut
into `block_size` blocks. No padding is wasted. The original notebook did this inside a batched
`datasets.map`, which dropped the ragged tail of **every 1,000-story batch**. `SequencePacker`
carries the remainder into the next batch, so only the final remainder of each split (< 512
tokens) is discarded. `PackingStats.validate()` checks the accounting identities on every run.

In [ ]:
from tinylm.data import load_packed_split, prepare_packed_data

sha = tokenizer_fingerprint(tok_dir)
if (Path(cfg.data.data_dir) / "meta.json").exists():
    from tinylm.utils import read_json

    meta = read_json(Path(cfg.data.data_dir) / "meta.json")
    assert meta["tokenizer_sha256"] == sha, "packed data was built with a different tokenizer"
else:
    meta = prepare_packed_data(cfg.data, tokenizer, sha)

print(
    f"{'split':<11}{'stories':>10}{'text tokens':>14}{'sequences':>11}{'dropped':>9}{'retained':>11}"
)
for split, s in meta["splits"].items():
    print(
        f"{split:<11}{s['examples']:>10,}{s['text_tokens']:>14,}{s['sequences']:>11,}"
        f"{s['discarded_tokens']:>9,}{100 * s['retained_fraction']:>10.4f}%"
    )

train_data = load_packed_split(cfg.data.data_dir, "train", sha)
val_data = load_packed_split(cfg.data.data_dir, "validation", sha)
print("\nfirst packed block, decoded (an <|endoftext|> mid-block is a story boundary):\n")
print(tokenizer.decode(train_data[0][:160].tolist()))

---
## Part 3: The model

| Design choice | Where it lives |
|---|---|
| Decoder-only transformer | `LlamaForCausalLM(LlamaConfig(...))`, built by `tinylm.modeling.build_model` |
| Pre-norm **RMSNorm** | built into the Llama architecture |
| **RoPE** positions | `rope_parameters={"rope_type": "default", "rope_theta": 10000.0}` |
| **SwiGLU** feed-forward | `hidden_act="silu"` + Llama's gated MLP |
| **Grouped-query attention** | 8 query heads share 2 key/value heads, so the KV cache is 4× smaller |
| No biases, tied embeddings | `attention_bias=False`, `mlp_bias=False`, `tie_word_embeddings=True` |
| Attention kernel | `attn_implementation="sdpa"`: PyTorch Scaled Dot Product Attention, which may dispatch to a fused kernel such as FlashAttention when the hardware, dtype and masking are eligible |

The model is created with `LlamaForCausalLM(config)`, **not** `from_pretrained`: every weight
starts random.

In [ ]:
from tinylm.modeling import build_model, initialization_loss, kv_cache_table, parameter_report
from tinylm.utils import seed_everything

seed_everything(cfg.training.seed)
model = build_model(cfg.model, tokenizer.eos_token_id, tokenizer.pad_token_id)
print(model.model.layers[0])
print()
print(parameter_report(model).format())

**Reading the layer:** `k_proj`/`v_proj` project to 2 heads × 64 dims (GQA), the MLP has three
matrices (SwiGLU), and RoPE has no module because it is applied inside the attention forward.

### Why GQA matters (and why it barely does here)
The KV cache stores keys and values for every past token during generation. Its size is
`2 × layers × batch × sequence × kv_heads × head_dim × bytes`.

In [ ]:
for row in kv_cache_table(cfg.model, batch_sizes=(1, 8, 64)):
    print(
        f"batch {row['batch_size']:>3} x {row['seq_len']} tokens: GQA {row['gqa_mb']:7.1f} MiB | "
        f"MHA {row['mha_mb']:7.1f} MiB  ({row['ratio']:.0f}x)"
    )

At batch size 1 the difference is a few MiB, so for a single-user demo of a 27M model, GQA saves
almost nothing. It matters when serving many sequences at once, and for larger models. It is
used here because it is part of the recipe under study; `scripts/run_ablation.py` measures its
quality cost at matched parameter count.

### Initialization sanity check
An untrained model should be close to uniform over the vocabulary, so its loss should be close
to ln(V). A much larger value means overconfident logits from a bad initialization scale. Checking
this costs one forward pass and catches the bug before hours of training.

In [ ]:
loss0, ln_v = initialization_loss(model, cfg.model.vocab_size)
print(
    f"initial loss {loss0:.4f} vs ln(vocab) {ln_v:.4f} -> {'OK' if abs(loss0 - ln_v) < 0.5 else 'CHECK INIT'}"
)

### Under the abstraction: the educational re-implementation
`tinylm.educational` re-implements RMSNorm, RoPE, grouped-query causal attention, SwiGLU and the
decoder block in plain PyTorch, using the same parameter names as Hugging Face Llama. So the
Hugging Face weights load directly into it, and both models produce the same logits:

In [ ]:
from tinylm.educational import ReferenceDecoder

ref = ReferenceDecoder(replace(cfg.model, architecture="reference"))
ref.load_state_dict(model.state_dict(), strict=True)
ids = torch.randint(0, cfg.model.vocab_size, (2, 32))
with torch.no_grad():
    diff = (model.eval()(input_ids=ids).logits - ref.eval()(ids).logits).abs().max().item()
print(f"max |logit difference| between HF Llama and the from-scratch decoder: {diff:.2e}")

---
## Part 4: Pretraining

The training loop (`tinylm.training.PretrainingRun`) is written out rather than delegated to
`transformers.Trainer`. Every technique is visible:

| Technique | Setting |
|---|---|
| AdamW, decay on matrices only | β = (0.9, 0.95), weight decay 0.1; norms and biases excluded |
| LR schedule | linear warmup, then cosine decay |
| Mixed precision | `torch.autocast` (bf16 / fp16 with `GradScaler`) |
| Gradient accumulation | micro-batch × accumulation = sequences per optimizer step |
| Gradient clipping | global norm 1.0 |
| Model selection | best **validation** loss → `<run>/best/` |
| Checkpoints | model, optimizer, scheduler, grad scaler, RNG states, data position |

### Resuming correctly
Colab sessions end. A checkpoint stores everything needed to continue **exactly**: the data
order is a function of (seed, sequences consumed), so a resumed run sees the same batches as an
uninterrupted one. `tests/test_checkpointing.py` checks bitwise equality on CPU. Metrics are kept
at two levels:

* **this invocation**: steps, time and tokens of the current process;
* **the experiment**: everything since step 0, accumulated across resumes.

The original notebook printed "Training finished in 0.4 minutes, final training loss 0.0001" after
resuming a nearly finished run. Those were invocation numbers, reported as if they described the
whole run.

Use `stop_after_steps` to train in session-sized pieces. Re-running the cell resumes from the
last checkpoint.

In [ ]:
from tinylm.training import PretrainingRun, training_summary

plan = training_summary(cfg, len(train_data), train_data.block_size)
print(
    f"{plan['tokens_per_step']:,} tokens/step x {plan['max_steps']:,} steps = "
    f"{plan['planned_training_tokens']:,} tokens ({plan['epochs']:.2f} epochs)"
)

STOP_AFTER_STEPS = None  # e.g. 2000 to train in pieces within a session limit
run = PretrainingRun(
    cfg,
    model,
    train_data,
    val_data,
    device,
    tokenizer,
    run_metadata={
        "tokenizer_sha256": sha,
        "dataset": meta["dataset"],
        "dataset_revision": meta["dataset_revision"],
    },
)
metrics = run.train(resume="auto", stop_after_steps=STOP_AFTER_STEPS)

last = metrics["invocations"][-1]
print(
    f"\nthis invocation : steps {last['start_step']} -> {last['end_step']}, {last['wall_clock_seconds']:.0f} s"
)
print(
    f"full experiment : {metrics['steps']} steps, {metrics['training_tokens']:,} tokens, "
    f"{metrics['train_seconds']:.0f} s of optimizer steps, best validation loss "
    f"{metrics['best_validation_loss']:.4f} at step {metrics['best_step']}"
)

---
## Part 5: Memory and throughput

Static training memory for a 27M model is small: fp32 weights, gradients and the two AdamW
moments together are about 4 × 4 bytes × 26.7M ≈ 408 MiB. Activations dominate, and they scale
with `micro_batch × block_size × hidden × layers`. That is why the first fix for an
out-of-memory error is a smaller micro-batch with more accumulation. It leaves the optimizer's
effective batch unchanged, and the trainer allows that change on resume.

Measured numbers come from `scripts/benchmark.py` and are recorded **per hardware** in
`results/benchmarks/`. Numbers from one GPU are never quoted for another.

In [ ]:
from tinylm.modeling import training_memory_estimate_mb

print(
    {
        k: f"{v:.0f} MiB"
        for k, v in training_memory_estimate_mb(parameter_report(model).total).items()
    }
)
if device.type == "cuda":
    print(
        f"peak allocated during this run: {metrics['peak_vram_allocated_mb']:.0f} MiB "
        f"(micro-batch {cfg.training.per_device_train_batch_size}, {resolve_precision(cfg.training.precision, device)})"
    )

---
## Part 6: Evaluation

### Training curves

In [ ]:
from IPython.display import Image, display

from tinylm.plotting import plot_training_history

out_dir = Path(cfg.training.output_dir)
display(
    Image(
        str(
            plot_training_history(
                out_dir / "training_history.jsonl",
                out_dir / "training_curves.png",
                cfg.model.vocab_size,
                cfg.name,
            )
        )
    )
)

The validation loss here is the **model-selection** metric: we looked at it to choose the
checkpoint, so it is slightly optimistic as an estimate of performance on new data.

### Held-out test loss: once, on the selected model
The selected model (`best/`) is evaluated once on the test split, which nothing above has touched.
Perplexity is `exp(loss)`, so it inherits every property of the loss: it depends on the tokenizer
(a different vocabulary changes the per-token loss) and on the dataset. It is only comparable
across runs that share both. There is no universal scale where a given perplexity means "good
stories".

In [ ]:
from tinylm.evaluation import evaluate_loss
from tinylm.modeling import load_model

best = load_model(out_dir / "best").to(device)
test_data = load_packed_split(cfg.data.data_dir, "test", sha)
test = evaluate_loss(
    best, test_data, cfg.training.eval_batch_size, device, "fp32", split="test", progress=True
)
print(
    f"TEST loss {test.loss:.4f} | perplexity {test.perplexity:.3f} | {test.predicted_tokens:,} predicted tokens"
)

### Generation
Loss measures next-token prediction on human-like data. It does not measure whether a
*generated* story is coherent. Generation evaluation complements it, and neither replaces the
other.

Decoding presets pass only the parameters their mode uses. Greedy gets no `temperature`, and
sampling presets set `top_k` explicitly, because leaving it unset makes transformers apply a
default top-k of 50. That default silently affected every sample in the original notebook.

In [ ]:
from tinylm.generation import PRESETS, generate

for name in ["greedy", "low_temperature", "balanced", "top_k", "creative"]:
    res = generate(best, tokenizer, "Once upon a time", name, max_new_tokens=120, seed=0)
    print(
        f"=== {name} {PRESETS[name].generation_kwargs()} | {res.new_tokens} tokens, "
        f"ended with EOS: {res.finished_with_eos}\n{res.text}\n"
    )

### Plausible stories do not prove generalization
A fluent story could be stitched together from memorized training text. Answering that question
takes measurement:

```bash
python scripts/evaluate.py --config <config>            # fixed prompt suite, seeds, lexical metrics
python scripts/check_memorization.py --config <config> --generations <run>/evaluation/generation_samples.jsonl
```

The memorization report checks every generated n-gram against the **entire** training set, finds
each generation's nearest training story, and runs the same analysis on held-out test stories as
a baseline (TinyStories is formulaic enough that real held-out stories share many n-grams with
training data). Known failure modes are documented with real outputs in
`results/failure_analysis.md`.

In [ ]:
from tinylm.evaluation import lexical_summary

records = [
    generate(best, tokenizer, "Once upon a time", "balanced", 150, seed=s).to_dict()
    for s in range(5)
]
print(json.dumps(lexical_summary(records), indent=2))

---
## Part 7: Save, reload, serve

`best/` is already a standard Hugging Face directory (`config.json`, `model.safetensors`,
tokenizer files, `generation_config.json`). The tied `lm_head.weight` is stored once as part of
the embedding, which is why some loaders mention it as "missing". The tie is restored on load,
and the tests verify identical logits after a round trip.

In [ ]:
from tinylm import TinyLM
from tinylm.modeling import weights_are_tied

lm = TinyLM.from_pretrained(out_dir / "best", device=str(device))
print("tied after reload:", weights_are_tied(lm.model))
print(
    lm.generate(
        "Once upon a time, a small rabbit", max_new_tokens=100, temperature=0.8, top_p=0.9, seed=42
    )
)

To serve it: `TINYLM_MODEL=<run>/best python app/app.py` starts the Gradio demo. Publishing to the
Hugging Face Hub is described in `model_card.md`.

## Where to go next
* Run the controlled architecture ablations: `python scripts/run_ablation.py --seeds 0 1 2`.
* Measure your hardware: `python scripts/benchmark.py --config configs/tiny-27m.yaml`.
* Read `docs/design-decisions.md` for the trade-offs behind each choice, and the README's
  limitations section before drawing conclusions from a 27M-parameter model.

### References
Vaswani et al. 2017 (Transformer) · Zhang & Sennrich 2019 (RMSNorm) · Su et al. 2021 (RoPE) ·
Shazeer 2020 (GLU variants / SwiGLU) · Ainslie et al. 2023 (GQA) · Dao et al. 2022 (FlashAttention) ·
Eldan & Li 2023 (TinyStories). Full citations are in the README.